# Bangla sentiment analysis: SentNoB benchmark and R-Drop ablations
This notebook produces measured results, not promised scores. Dataset input:
`/kaggle/input/datasets/rahatrihan/sentnbangla/SentNoB Dataset`.
The supplied Google TensorFlow1 uncased BERT path is NOT the csebuetnlp BanglaBERT
checkpoint. We do not silently substitute it. Enable Kaggle Internet and a GPU,
or attach complete Hugging Face model directories and set MODEL_SOURCES below.

Full mode: five seeds, two learning-rate trials per transformer method, up to six
epochs per run. This is a substantial GPU experiment and can exceed one session.
Completed runs resume from this output directory; interrupted epochs restart.
Persist/download the outputs before ending a Kaggle session. Smoke mode verifies
execution only and is not suitable for a paper.

Method scope: R-Drop is an existing NeurIPS 2021 method, not a new invention.
SentNoB has no gold noise-type annotations, so there is no auxiliary noise head.
No synthetic text, assumed noise labels, fabricated scores, or test-set tuning.
References:
- https://aclanthology.org/2021.findings-emnlp.278/ (SentNoB)
- https://aclanthology.org/2022.findings-naacl.98/ (BanglaBERT)
- https://aclanthology.org/2024.wnut-1.5/ (NC-SentNoB; different protocol)
- https://arxiv.org/abs/2106.14448 (R-Drop)
- https://huggingface.co/datasets/khondoker/SentNoB/blob/main/README.md (labels)


In [ ]:
# Run this setup cell FIRST in a fresh Kaggle session. Internet is required for
# installation/downloads. We retain Kaggle's installed CUDA-compatible PyTorch.
import subprocess, sys, tempfile, importlib.util
from importlib import metadata
from pathlib import Path
# Start with a fresh Kaggle runtime if the previous installer already downgraded
# packages. A kernel restart alone does not restore installed package versions.
required = ['transformers','safetensors','scikit-learn','matplotlib','seaborn',
            'pandas','sentencepiece','openpyxl','joblib','tqdm']
installed = {d.metadata['Name'].lower(): d.version for d in metadata.distributions() if d.metadata['Name']}
missing = [p for p in required if p.lower() not in installed]
if importlib.util.find_spec('normalizer') is None:
    missing.append('git+https://github.com/csebuetnlp/normalizer.git')
if missing:
    constraints = Path(tempfile.gettempdir())/'bangla_preserve_environment.txt'
    constraints.write_text('\n'.join(f'{name}=={version}' for name,version in sorted(installed.items())))
    subprocess.check_call([sys.executable,'-m','pip','install','-q',
                           '--constraint',str(constraints),*missing])
print('Setup preserves installed packages. Installed only missing dependencies:',missing)


## 1. Configuration
Default numeric labels follow the original SentNoB card: 0=neutral, 1=positive,
2=negative. If your dataset was recoded, change NUMERIC_LABEL_MAP accordingly.
FILES can specify exact train/validation/test filenames when autodetection is
ambiguous. We never fabricate a random split when official splits are absent.
Clean mode removes exact normalized overlap from training/validation while
preserving test rows. This changes the training protocol: literature comparisons
remain contextual. Near-duplicate candidates are audited, not declared identical.


In [ ]:
from pathlib import Path
import os, json, re, random, hashlib, math, time, gc, shutil, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, log_loss, precision_recall_curve,
    average_precision_score)
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import NearestNeighbors
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup, set_seed)
from normalizer import normalize
import joblib
from tqdm.auto import tqdm

CODE_VERSION = 'sentnob-rdrop-1.1.0'  # Increment after changing training code.
DATA_ROOT = Path('/kaggle/input/datasets/rahatrihan/sentnbangla/SentNoB Dataset')
SUPPLIED_ENGLISH_TF_MODEL = Path('/kaggle/input/models/google/bert/tensorflow1/bert-uncased-l-10-h-128-a-2/2')
FILES = {}  # e.g. {'train':'Train.csv', 'validation':'Val.csv', 'test':'Test.csv'}
TEXT_COLUMN = None
LABEL_COLUMN = None
NUMERIC_LABEL_MAP = {0: 0, 1: 1, 2: 2}
CLASS_NAMES = ['neutral', 'positive', 'negative']
LABELS = [0, 1, 2]
MODE = 'full'  # 'smoke' runs a small integration check; never publish those scores.
CLEAN_EXACT_OVERLAP = True
SEEDS = [13, 42, 87, 101, 2026] if MODE == 'full' else [13]
LR_GRID = [1e-5, 2e-5] if MODE == 'full' else [2e-5]
MAX_EPOCHS = 6 if MODE == 'full' else 1
PATIENCE = 2
MAX_LENGTH = 128
BATCH_SIZE = 8
ACCUMULATION_STEPS = 4
R_DROP_ALPHA = 1.0  # Fixed a priori, not chosen using test results.
WEIGHT_DECAY = 0.01
BOOTSTRAP_REPEATS = 2000 if MODE == 'full' else 100
MODEL_SOURCES = {
    'banglabert': 'csebuetnlp/banglabert',
    'mbert': 'google-bert/bert-base-multilingual-cased',
}
# Set these to saved commit hashes from a previous manifest for exact reruns.
MODEL_REVISIONS = {'banglabert': 'main', 'mbert': 'main'}
METHODS = {
    'mBERT': {'encoder': 'mbert', 'weighted': False, 'rdrop': False},
    'BanglaBERT': {'encoder': 'banglabert', 'weighted': False, 'rdrop': False},
    'BanglaBERT_weighted': {'encoder': 'banglabert', 'weighted': True, 'rdrop': False},
    'BanglaBERT_RDrop': {'encoder': 'banglabert', 'weighted': False, 'rdrop': True},
    'BanglaBERT_weighted_RDrop': {'encoder': 'banglabert', 'weighted': True, 'rdrop': True},
}
OUT = Path('/kaggle/working/bangla_sentiment_results_v2')
OUT.mkdir(parents=True, exist_ok=True)
for sub in ['figures', 'tables', 'audit', 'runs', 'checkpoints']:
    (OUT / sub).mkdir(exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a GPU accelerator in Kaggle before the full run.')
print('GPU:', torch.cuda.get_device_name(0), '| mode:', MODE)
print('The supplied TensorFlow1 English checkpoint is not used:', SUPPLIED_ENGLISH_TF_MODEL)
print('Full run uses one GPU; a second GPU is not required.')
sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)

def save_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def save_fig(name):
    plt.tight_layout()
    for ext in ['png', 'pdf']:
        plt.savefig(OUT / 'figures' / f'{name}.{ext}', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

def basic_normalize(text):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(text))).strip()

def macro_f1(y, p):
    return float(f1_score(y, p, labels=LABELS, average='macro', zero_division=0))

def scores(y, p):
    return {'accuracy': float(accuracy_score(y, p)), 'macro_f1': macro_f1(y, p),
            'micro_f1': float(f1_score(y, p, average='micro')),
            'weighted_f1': float(f1_score(y, p, average='weighted', zero_division=0))}


## 2. Dataset discovery, validation and leakage audit
Numeric labels are not inferred from sentence meaning. Unrecognized/missing
labels cause explicit errors. Contradictory training/dev groups are quarantined
using labels within that split only; test rows and labels are preserved.
No sentiment examples from the test set are printed during development.


In [ ]:
def split_name(path):
    tokens = re.split(r'[^a-z]+', path.stem.lower())
    if 'train' in tokens or 'training' in tokens: return 'train'
    if any(x in tokens for x in ['val', 'valid', 'validation', 'dev']): return 'validation'
    if 'test' in tokens or 'testing' in tokens: return 'test'
    return None

def read_table(path):
    if path.suffix.lower() in ['.xlsx', '.xls']:
        return pd.read_excel(path)
    return pd.read_csv(path, sep='\t' if path.suffix.lower() == '.tsv' else ',',
                       encoding='utf-8-sig')

def choose_column(columns, override, candidates, kind):
    if override:
        if override not in columns: raise ValueError(f'Missing {kind} column: {override}')
        return override
    found = [c for c in columns if c.strip().lower() in candidates]
    if len(found) != 1:
        raise ValueError(f'Ambiguous {kind} columns {list(columns)}. Set the column override.')
    return found[0]

def encode_label(value):
    aliases = {'neutral': 0, 'positive': 1, 'negative': 2,
               'নিরপেক্ষ': 0, 'ইতিবাচক': 1, 'নেতিবাচক': 2}
    s = str(value).strip().lower()
    if s in aliases: return aliases[s]
    try:
        n = float(s)
        if not math.isfinite(n) or n != int(n): raise ValueError()
        return NUMERIC_LABEL_MAP[int(n)]
    except (ValueError, KeyError, OverflowError):
        raise ValueError(f'Unknown label {value!r}; check NUMERIC_LABEL_MAP.')

def load_splits(root, overrides):
    if not root.exists(): raise FileNotFoundError(f'Attach the dataset at {root}')
    candidates = sorted(p for p in root.rglob('*') if p.suffix.lower() in ['.csv','.tsv','.xlsx','.xls'])
    print('Available data files:', [str(p.relative_to(root)) for p in candidates])
    selected = {}
    for split in ['train', 'validation', 'test']:
        options = [root / overrides[split]] if split in overrides else [p for p in candidates if split_name(p) == split]
        if len(options) != 1:
            raise ValueError(f'{split}: found {len(options)} possible files. Set FILES explicitly.')
        selected[split] = options[0]
    frames, manifests = {}, {}
    for split, path in selected.items():
        df = read_table(path)
        df.columns = [str(c).strip() for c in df.columns]
        tc = choose_column(df.columns, TEXT_COLUMN, ['data','text','sentence','comment','content','review'], 'text')
        lc = choose_column(df.columns, LABEL_COLUMN, ['label','sentiment','polarity','class','sentiment_label'], 'label')
        if df[[tc,lc]].isna().any().any(): raise ValueError(f'Missing text/labels in {path}; review source.')
        df = pd.DataFrame({'id': [f'{split}:{i}' for i in range(len(df))],
                           'text': df[tc].astype(str), 'label': df[lc].map(encode_label)})
        df['key'] = df.text.map(basic_normalize)
        if (df.key == '').any(): raise ValueError(f'Empty text in {path}.')
        if set(df.label) != set(LABELS): raise ValueError(f'{split} must contain all three classes.')
        frames[split] = df
        manifests[split] = {'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                            'original_rows': len(df), 'text_column': tc, 'label_column': lc}
    return frames, manifests

def quarantine_conflicts(frames, key_column, stage):
    """Never resolve conflicts by majority vote or by consulting test labels."""
    combined = pd.concat([df.assign(split=s) for s,df in frames.items()], ignore_index=True)
    all_counts = combined.groupby(key_column).label.nunique()
    global_bad = set(all_counts[all_counts > 1].index)
    combined[combined[key_column].isin(global_bad)][['id','split','label']].to_csv(
        OUT/'audit'/f'{stage}_all_conflicting_ids.csv',index=False)
    summary = []
    cleaned = {}
    for split,df in frames.items():
        counts = df.groupby(key_column).label.nunique()
        bad = set(counts[counts > 1].index)
        mask = df[key_column].isin(bad)
        # Decisions use this split's labels only. Cross-split overlap removal below
        # uses text equality regardless of the corresponding sentiment labels.
        action = 'retained_original_test' if split == 'test' else 'quarantined'
        df.loc[mask,['id','label']].to_csv(OUT/'audit'/f'{stage}_{split}_conflicting_ids.csv',index=False)
        cleaned[split] = (df if split == 'test' else df.loc[~mask]).copy().reset_index(drop=True)
        summary.append({'split':split,'conflicting_groups':len(bad),'affected_rows':int(mask.sum()),'action':action})
    report = pd.DataFrame(summary)
    report.to_csv(OUT/'audit'/f'{stage}_conflict_summary.csv',index=False)
    print(f'{stage} conflict handling:\n'+report.to_string(index=False))
    return cleaned

frames, input_manifest = load_splits(DATA_ROOT, FILES)
frames = quarantine_conflicts(frames,'key','raw')
audit = []
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    overlap = set(frames[a].key) & set(frames[b].key)
    audit.append({'split_a':a, 'split_b':b, 'overlapping_unique_texts':len(overlap)})
pd.DataFrame(audit).to_csv(OUT/'audit/exact_overlap.csv', index=False)
print(pd.DataFrame(audit).to_string(index=False))
if CLEAN_EXACT_OVERLAP:
    heldout = set(frames['validation'].key) | set(frames['test'].key)
    frames['train'] = frames['train'][~frames['train'].key.isin(heldout)].drop_duplicates('key')
    frames['validation'] = frames['validation'][~frames['validation'].key.isin(set(frames['test'].key))].drop_duplicates('key')
for split, df in frames.items():
    if set(df.label) != set(LABELS): raise ValueError(f'Cleaning left missing classes in {split}.')
    frames[split] = df.reset_index(drop=True)
    input_manifest[split]['used_rows_before_smoke'] = len(df)
    df[['id','label']].to_csv(OUT/'audit'/f'{split}_membership.csv', index=False)
    if MODE == 'smoke':
        frames[split] = df.groupby('label', group_keys=False).sample(n=min(24, int(df.label.value_counts().min())), random_state=13).reset_index(drop=True)

# The same deterministic pretrained-model normalization is used in every method.
for split in frames:
    frames[split]['model_text'] = frames[split].text.map(lambda x: basic_normalize(normalize(x)))
    if (frames[split].model_text == '').any():
        raise ValueError(f'Official normalization produced empty input in {split}. Review these rows.')
# Normalization can cause extra collisions. Audit and remove from training/dev,
# preserving the original test labels, including any remaining contradictions.
frames = quarantine_conflicts(frames,'model_text','normalized')
if CLEAN_EXACT_OVERLAP:
    heldout = set(frames['validation'].model_text) | set(frames['test'].model_text)
    frames['train'] = frames['train'][~frames['train'].model_text.isin(heldout)].drop_duplicates('model_text')
    frames['validation'] = frames['validation'][~frames['validation'].model_text.isin(set(frames['test'].model_text))].drop_duplicates('model_text')
for split in frames:
    frames[split] = frames[split].reset_index(drop=True)
    if set(frames[split].label) != set(LABELS): raise ValueError(f'Invalid final {split} class coverage.')
    input_manifest[split]['final_rows'] = len(frames[split])
    frames[split][['id','label']].to_csv(OUT/'audit'/f'{split}_membership.csv', index=False)
train, val, test = [frames[s] for s in ['train','validation','test']]

# Nearest-neighbor text audit; only IDs and similarity are exported, not test text.
audit_vec = TfidfVectorizer(analyzer='char', ngram_range=(3,5), max_features=80000, dtype=np.float32)
audit_x = audit_vec.fit_transform(train.model_text)
nn = NearestNeighbors(n_neighbors=1, metric='cosine', algorithm='brute').fit(audit_x)
near = []
for split in ['validation','test']:
    for start in range(0,len(frames[split]),128):
        part = frames[split].iloc[start:start+128]
        dist, ind = nn.kneighbors(audit_vec.transform(part.model_text))
        for row, d, j in zip(part.itertuples(), dist[:,0], ind[:,0]):
            if 1-d >= 0.92:
                near.append({'heldout_id':row.id,'train_id':train.iloc[j].id,'similarity':float(1-d)})
pd.DataFrame(near, columns=['heldout_id','train_id','similarity']).to_csv(OUT/'audit/near_duplicate_candidates.csv', index=False)
print('Near-duplicate candidates requiring review:',len(near))
del audit_vec, audit_x, nn
gc.collect()


## 3. Dataset figures and reproducibility manifest
Figures use English class names to avoid missing Bangla glyphs in plot fonts.
Macro-F1 is the predeclared model-selection metric. Hyperparameter tuning sees
training and validation labels only. Literature scores are never optimization targets.


In [ ]:
counts = pd.DataFrame({s:df.label.value_counts().reindex(LABELS,fill_value=0) for s,df in frames.items()})
counts.index = CLASS_NAMES
counts.to_csv(OUT/'tables/class_counts.csv')
counts.plot.bar(figsize=(8,4)); plt.ylabel('Examples'); plt.xticks(rotation=0); save_fig('01_class_distribution')
plt.figure(figsize=(8,4))
for s,df in frames.items():
    sns.ecdfplot(df.text.str.split().str.len(), label=s)
plt.xlabel('Whitespace-delimited word count'); plt.ylabel('Cumulative fraction'); plt.legend(); save_fig('02_text_lengths')

config = dict(code_version=CODE_VERSION, mode=MODE, seeds=SEEDS, lr_grid=LR_GRID,
              max_epochs=MAX_EPOCHS, patience=PATIENCE, max_length=MAX_LENGTH,
              batch_size=BATCH_SIZE, accumulation=ACCUMULATION_STEPS,
              rdrop_alpha=R_DROP_ALPHA, weight_decay=WEIGHT_DECAY,
              clean_overlap=CLEAN_EXACT_OVERLAP, labels=NUMERIC_LABEL_MAP,
              sources=MODEL_SOURCES, revisions=MODEL_REVISIONS, methods=METHODS)
fingerprint = hashlib.sha256(json.dumps([config,input_manifest],sort_keys=True).encode()).hexdigest()
manifest_path = OUT/'manifest.json'
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    if old['fingerprint'] != fingerprint:
        raise ValueError('Configuration/data changed. Set OUT to a new output directory to avoid stale results.')
save_json(manifest_path, {'fingerprint':fingerprint,'config':config,'inputs':input_manifest,
    'gpu':torch.cuda.get_device_name(0),'near_duplicate_candidates':len(near),
    'protocol':'cleaned training/dev, original test' if CLEAN_EXACT_OVERLAP else 'supplied splits retained',
    'literature_comparability':'unverified; no direct superiority claim',
    'notes':'Contradictory train/dev groups quarantined within each split; test labels preserved. R-Drop is established; no gold noise labels supplied.'})
(OUT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))


## 4. Train/validation tuning for traditional baselines
Character features help capture spelling variation. C is selected on validation
macro-F1. SVM decision scores are NOT treated as calibrated probabilities.


In [ ]:
def lexical_pipeline(kind, c):
    features = FeatureUnion([
        ('word',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=60000,sublinear_tf=True)),
        ('char',TfidfVectorizer(analyzer='char',ngram_range=(3,5),min_df=2,max_features=100000,sublinear_tf=True))])
    clf = (LogisticRegression(C=c,class_weight='balanced',max_iter=2000,random_state=13)
           if kind == 'TFIDF_LogReg' else LinearSVC(C=c,class_weight='balanced',random_state=13,max_iter=10000))
    return Pipeline([('features',features),('clf',clf)])

lexical_models, lexical_tuning = {}, []
for kind in ['TFIDF_LogReg','TFIDF_SVM']:
    best_score, best_pipe = -1, None
    for c in [0.5,1.0,2.0]:
        pipe = lexical_pipeline(kind,c)
        pipe.fit(train.model_text,train.label)
        score = macro_f1(val.label,pipe.predict(val.model_text))
        lexical_tuning.append({'method':kind,'C':c,'validation_macro_f1':score})
        if score > best_score: best_score,best_pipe = score,pipe
    lexical_models[kind] = best_pipe
    joblib.dump(best_pipe,OUT/'checkpoints'/f'{kind}.joblib')
pd.DataFrame(lexical_tuning).to_csv(OUT/'tables/lexical_tuning.csv',index=False)


## 5. Transformer training implementation
R-Drop performs two stochastic forward passes of the SAME text and minimizes
symmetric KL divergence plus average supervised loss. This is dropout consistency,
not spelling correction. All methods share tokenization, max length, search size,
seeds, early stopping, and evaluation. R-Drop uses more computation; runtime is saved.


In [ ]:
tokenizers, encoded, resolved = {}, {}, {}
class EncodedDataset(Dataset):
    def __init__(self, enc, labels): self.enc,self.labels = enc,list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        return {**{k:v[i] for k,v in self.enc.items()},'labels':self.labels[i]}

for enc_name,source in MODEL_SOURCES.items():
    revision = MODEL_REVISIONS[enc_name]
    local = Path(source).is_dir()
    kwargs = {} if local else {'revision':revision}
    tok = AutoTokenizer.from_pretrained(source, **kwargs)
    tokenizers[enc_name] = tok
    encoded[enc_name] = {}
    truncation = []
    for s,df in frames.items():
        enc = tok(df.model_text.tolist(),truncation=True,max_length=MAX_LENGTH)
        encoded[enc_name][s] = EncodedDataset(enc,df.label)
        lengths = [len(x) for x in tok(df.model_text.tolist(),truncation=False)['input_ids']]
        ids = [i for row in enc['input_ids'] for i in row]
        unk = sum(i == tok.unk_token_id for i in ids) / max(1,len(ids))
        truncation.append({'encoder':enc_name,'split':s,'truncated_fraction':float(np.mean(np.array(lengths)>MAX_LENGTH)),
                           'unknown_token_fraction':unk})
    pd.DataFrame(truncation).to_csv(OUT/'audit'/f'{enc_name}_tokenization.csv',index=False)
    model_probe = AutoModelForSequenceClassification.from_pretrained(source,num_labels=3,**kwargs)
    if enc_name == 'banglabert' and model_probe.config.model_type != 'electra':
        raise ValueError('Expected the csebuetnlp BanglaBERT ELECTRA checkpoint. Check MODEL_SOURCES.')
    resolved[enc_name] = getattr(model_probe.config,'_commit_hash',None) or str(source)
    del model_probe
revision_path = OUT/'resolved_model_revisions.json'
if revision_path.exists() and json.loads(revision_path.read_text()) != resolved:
    raise ValueError('Model revisions changed. Pin MODEL_REVISIONS or use a new output directory.')
save_json(revision_path,resolved)

def loader(enc_name,split,shuffle=False,seed=13):
    tok = tokenizers[enc_name]
    def collate(rows):
        labels = torch.tensor([r['labels'] for r in rows],dtype=torch.long)
        inputs = [{k:v for k,v in r.items() if k != 'labels'} for r in rows]
        return dict(tok.pad(inputs,padding=True,return_tensors='pt'),labels=labels)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(encoded[enc_name][split],batch_size=BATCH_SIZE,shuffle=shuffle,
                      collate_fn=collate,num_workers=0,pin_memory=True,generator=generator)

def predict(model,enc_name,split):
    model.eval(); result = []
    with torch.inference_mode():
        for batch in loader(enc_name,split):
            batch.pop('labels')
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            with torch.autocast(device_type='cuda',dtype=torch.float16): logits = model(**batch).logits
            result.append(logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(result)

def symmetric_kl(a,b):
    a,b = a.float(),b.float()
    return 0.5*(F.kl_div(F.log_softmax(a,-1),F.softmax(b,-1),reduction='batchmean')+
                F.kl_div(F.log_softmax(b,-1),F.softmax(a,-1),reduction='batchmean'))

def train_run(method,lr,seed,phase):
    spec = METHODS[method]; enc_name = spec['encoder']
    run_dir = OUT/'runs'/f'{phase}_{method}_lr{lr:g}_seed{seed}'
    run_dir.mkdir(exist_ok=True)
    completion = run_dir/'complete.json'
    if completion.exists():
        meta = json.loads(completion.read_text())
        if meta['fingerprint'] != fingerprint: raise ValueError('Stale completed run.')
        if (run_dir/'best_model/config.json').exists():
            print('Resume completed run:',run_dir.name)
            return meta,run_dir
    set_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    source = MODEL_SOURCES[enc_name]
    revision = resolved[enc_name]
    kwargs = {} if Path(source).is_dir() else {'revision':revision}
    model = AutoModelForSequenceClassification.from_pretrained(source,num_labels=3,
        id2label=dict(enumerate(CLASS_NAMES)),label2id={x:i for i,x in enumerate(CLASS_NAMES)},**kwargs).to(DEVICE)
    dl = loader(enc_name,'train',True,seed)
    total_steps = math.ceil(len(dl)/ACCUMULATION_STEPS)*MAX_EPOCHS
    optimizer = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(optimizer,int(0.1*total_steps),total_steps)
    scaler = torch.amp.GradScaler('cuda')
    class_counts = np.bincount(train.label,minlength=3)
    weights = torch.tensor(len(train)/(3*class_counts),dtype=torch.float32,device=DEVICE) if spec['weighted'] else None
    best,stale,history = -1.0,0,[]
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); optimizer.zero_grad(set_to_none=True); loss_sum,n_seen = 0.0,0
        for step,batch in enumerate(tqdm(dl,desc=f'{phase} {method} seed={seed} epoch={epoch}',leave=False)):
            labels = batch.pop('labels').to(DEVICE)
            batch = {k:v.to(DEVICE) for k,v in batch.items()}
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                a = model(**batch).logits
                loss = F.cross_entropy(a.float(),labels,weight=weights)
                if spec['rdrop']:
                    b = model(**batch).logits
                    loss = 0.5*(loss+F.cross_entropy(b.float(),labels,weight=weights))+R_DROP_ALPHA*symmetric_kl(a,b)
            loss_sum += loss.item()*len(labels); n_seen += len(labels)
            # Correct denominator for the final incomplete accumulation group.
            group_start = (step//ACCUMULATION_STEPS)*ACCUMULATION_STEPS
            group_size = min(ACCUMULATION_STEPS,len(dl)-group_start)
            scaler.scale(loss/group_size).backward()
            if (step+1)%ACCUMULATION_STEPS == 0 or step+1 == len(dl):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= previous_scale: scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        probs = predict(model,enc_name,'validation')
        metric = macro_f1(val.label,probs.argmax(1))
        history.append({'epoch':epoch,'train_objective':loss_sum/n_seen,
                        'validation_macro_f1':metric,'validation_cross_entropy':log_loss(val.label,probs,labels=LABELS)})
        pd.DataFrame(history).to_csv(run_dir/'history.csv',index=False)
        print(run_dir.name,'epoch',epoch,'validation macro-F1',round(metric,4),flush=True)
        if metric > best:
            best,stale = metric,0
            model.save_pretrained(run_dir/'best_model',safe_serialization=True)
            tokenizers[enc_name].save_pretrained(run_dir/'best_model')
            np.save(run_dir/'validation_probabilities.npy',probs)
        else:
            stale += 1
            if stale >= PATIENCE: break
    meta = {'method':method,'lr':lr,'seed':seed,'phase':phase,'validation_macro_f1':best,
            'epochs':len(history),'seconds':time.perf_counter()-started,
            'peak_gpu_gb':torch.cuda.max_memory_allocated()/1024**3,
            'fingerprint':fingerprint,'model_revision':revision}
    save_json(completion,meta)
    del model,optimizer,scheduler,scaler
    gc.collect(); torch.cuda.empty_cache()
    return meta,run_dir


## 6. Freeze hyperparameters before any model sees test labels
Selection uses one tuning seed (13); five final seeds measure training variability.
Alpha is fixed, so every transformer method gets the same two learning-rate trials.
Checkpoint/early-stop selection uses validation macro-F1 only. Test-set performance
must not be used to change these settings after inspecting the final tables.


In [ ]:
tuning_rows, selected_lrs = [], {}
for method in METHODS:
    candidates = []
    for lr in LR_GRID:
        meta,run_dir = train_run(method,lr,13,'tune')
        candidates.append(meta); tuning_rows.append(meta)
    selected_lrs[method] = max(candidates,key=lambda r:r['validation_macro_f1'])['lr']
pd.DataFrame(tuning_rows).to_csv(OUT/'tables/transformer_tuning.csv',index=False)
save_json(OUT/'frozen_selection.json',{'selected_lrs':selected_lrs,'seeds':SEEDS,
    'primary_metric':'macro_f1','primary_comparison':['BanglaBERT_weighted_RDrop','BanglaBERT'],
    'test_used_for_selection':False})
print('Hyperparameters frozen:',selected_lrs)


## 7. Final runs and predictions
Checkpoints are retained for each final run. If disk space becomes tight, move
completed results to persistent storage before continuing. Test results are not
printed until the complete predeclared suite finishes. Progress histories and
per-run files are saved as training proceeds.


In [ ]:
prediction_records, result_rows = [], []
for kind,pipe in lexical_models.items():
    p = pipe.predict(test.model_text)
    row = dict(method=kind,seed=13,validation_macro_f1=macro_f1(val.label,pipe.predict(val.model_text)),**scores(test.label,p))
    result_rows.append(row)
    rec = test[['id','label','key']].copy(); rec['method']=kind; rec['seed']=13; rec['prediction']=p
    if kind == 'TFIDF_LogReg':
        prob = pipe.predict_proba(test.model_text)
        assert np.array_equal(pipe.classes_,LABELS)
        for j in LABELS: rec[f'p{j}']=prob[:,j]
    prediction_records.append(rec)

final_runs = []
for method in METHODS:
    for seed in SEEDS:
        lr = selected_lrs[method]
        # Reuse the completed chosen trial for seed 13; identical hyperparameters.
        phase = 'tune' if seed == 13 else 'final'
        meta,run_dir = train_run(method,lr,seed,phase)
        prob_path = run_dir/'test_probabilities.npy'
        if prob_path.exists(): probs = np.load(prob_path)
        else:
            model = AutoModelForSequenceClassification.from_pretrained(run_dir/'best_model').to(DEVICE)
            probs = predict(model,METHODS[method]['encoder'],'test')
            np.save(prob_path,probs)
            del model; gc.collect(); torch.cuda.empty_cache()
        assert probs.shape == (len(test),3)
        result_rows.append(dict(meta,**scores(test.label,probs.argmax(1))))
        rec = test[['id','label','key']].copy(); rec['method']=method; rec['seed']=seed; rec['prediction']=probs.argmax(1)
        for j in LABELS: rec[f'p{j}']=probs[:,j]
        prediction_records.append(rec)
        final_runs.append(dict(meta,path=str(run_dir)))
        save_json(OUT/'runs/final_progress.json',final_runs)

results = pd.DataFrame(result_rows)
predictions = pd.concat(prediction_records,ignore_index=True)
results.to_csv(OUT/'tables/results_per_seed.csv',index=False)
predictions.to_csv(OUT/'tables/test_predictions.csv',index=False)
summary = results.groupby('method')[['accuracy','macro_f1','micro_f1','weighted_f1']].agg(['mean','std'])
summary.to_csv(OUT/'tables/results_summary.csv')
print(summary.round(4).to_string())
print('Standard deviation is undefined for single deterministic traditional baselines; it is not zero.')


## 8. Figures, per-class reports and statistical comparisons
All bar error bars show sample SD over seeds, not confidence intervals.
Confusion matrices, PR and calibration plots use each method's validation-selected
seed, not the best test seed. Bootstrap compares the predeclared full method with
plain BanglaBERT, resampling both matched seeds and duplicate-text clusters.
This interval describes this dataset and training randomness, not all Bangla text.


In [ ]:
order = ['TFIDF_LogReg','TFIDF_SVM']+list(METHODS)
means = results.groupby('method').macro_f1.mean().reindex(order)
stds = results.groupby('method').macro_f1.std().reindex(order).fillna(0)
plt.figure(figsize=(10,5)); plt.barh(order,means,xerr=stds,capsize=3)
plt.xlabel('Test macro-F1 (mean ± seed SD)'); plt.xlim(0,1); save_fig('03_model_comparison')
ablation_order = [x for x in METHODS if x.startswith('BanglaBERT')]
plt.figure(figsize=(9,4)); plt.barh(ablation_order,means.reindex(ablation_order),xerr=stds.reindex(ablation_order),capsize=3)
plt.xlabel('Test macro-F1 (mean ± seed SD)'); plt.xlim(0,1); save_fig('04_ablation')

chosen = results.sort_values(['method','validation_macro_f1','seed'],ascending=[True,False,True]).drop_duplicates('method')
perclass, slice_rows, calibration_rows, probability_metrics = [], [], [], []
for row in chosen.itertuples():
    rec = predictions[(predictions.method==row.method)&(predictions.seed==row.seed)].copy().reset_index(drop=True)
    assert rec.id.tolist() == test.id.tolist()
    report = classification_report(rec.label,rec.prediction,labels=LABELS,target_names=CLASS_NAMES,output_dict=True,zero_division=0)
    save_json(OUT/'tables'/f'{row.method}_classification_report.json',report)
    for c in CLASS_NAMES: perclass.append(dict(method=row.method,sentiment=c,**report[c]))
    fig,ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay.from_predictions(rec.label,rec.prediction,labels=LABELS,display_labels=CLASS_NAMES,
                                            normalize='true',values_format='.2f',cmap='Blues',ax=ax,colorbar=False)
    ax.set_title(f'{row.method}\nValidation-selected seed {row.seed}')
    save_fig(f'05_confusion_{row.method}')
    if all(c in rec and rec[c].notna().all() for c in ['p0','p1','p2']):
        prob = rec[['p0','p1','p2']].to_numpy()
        plt.figure(figsize=(6,4))
        for j,c in enumerate(CLASS_NAMES):
            precision,recall,_ = precision_recall_curve((rec.label==j).astype(int),prob[:,j])
            ap = average_precision_score((rec.label==j).astype(int),prob[:,j])
            plt.plot(recall,precision,label=f'{c}: AP={ap:.3f}')
        plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); save_fig(f'06_pr_{row.method}')
        conf = prob.max(1); correct = (prob.argmax(1)==rec.label.to_numpy()).astype(float)
        bins = np.minimum((conf*10).astype(int),9); ece=0
        xs,ys = [],[]
        for b in range(10):
            mask = bins==b
            if not mask.any(): continue
            avg_conf,acc = conf[mask].mean(),correct[mask].mean()
            ece += mask.mean()*abs(avg_conf-acc); xs.append(avg_conf); ys.append(acc)
            calibration_rows.append({'method':row.method,'bin':b,'n':int(mask.sum()),'confidence':avg_conf,'accuracy':acc})
        probability_metrics.append({'method':row.method,'selected_seed':row.seed,'ece_10_bins':ece,
            'log_loss':log_loss(rec.label,prob,labels=LABELS),
            'multiclass_brier':float(np.mean(np.sum((prob-np.eye(3)[rec.label])**2,axis=1)))})
        plt.figure(figsize=(5,4)); plt.plot([0,1],[0,1],'--',color='gray'); plt.plot(xs,ys,'o-')
        plt.xlabel('Mean confidence'); plt.ylabel('Empirical accuracy'); plt.title(row.method); save_fig(f'07_calibration_{row.method}')
    text = test.text
    slices = {'contains_negation':text.str.contains(r'(?<!\S)(?:না|নয়|নয়|নেই)(?!\S)',regex=True),
              'contains_Latin':text.str.contains('[A-Za-z]',regex=True),
              'longer_than_30_words':text.str.split().str.len()>30}
    for name,mask in slices.items():
        if mask.sum():
            slice_rows.append({'method':row.method,'slice':name,'n':int(mask.sum()),
                               'classes_present':int(rec.loc[mask,'label'].nunique()),**scores(rec.loc[mask,'label'],rec.loc[mask,'prediction'])})
pd.DataFrame(perclass).to_csv(OUT/'tables/per_class.csv',index=False)
pd.DataFrame(slice_rows).to_csv(OUT/'tables/heuristic_slices.csv',index=False)
pd.DataFrame(calibration_rows).to_csv(OUT/'tables/calibration_bins.csv',index=False)
pd.DataFrame(probability_metrics).to_csv(OUT/'tables/probability_metrics.csv',index=False)
pc = pd.DataFrame(perclass).pivot(index='method',columns='sentiment',values='f1-score').reindex(order)
plt.figure(figsize=(7,5)); sns.heatmap(pc,annot=True,fmt='.3f',vmin=0,vmax=1,cmap='Blues'); save_fig('08_per_class_f1')

for method in METHODS:
    fig,axes = plt.subplots(1,2,figsize=(10,4))
    for r in final_runs:
        if r['method'] != method: continue
        hist = pd.read_csv(Path(r['path'])/'history.csv')
        axes[0].plot(hist.epoch,hist.train_objective,label=str(r['seed']))
        axes[1].plot(hist.epoch,hist.validation_macro_f1,label=str(r['seed']))
    axes[0].set(xlabel='Epoch',ylabel='Training objective')
    axes[1].set(xlabel='Epoch',ylabel='Validation macro-F1')
    axes[1].legend(title='Seed'); fig.suptitle(method); save_fig(f'09_learning_{method}')

costs = pd.DataFrame(final_runs).groupby('method')[['seconds','peak_gpu_gb','epochs']].agg(['mean','std'])
costs.to_csv(OUT/'tables/transformer_training_cost.csv')
fig,axes = plt.subplots(1,2,figsize=(11,4),sharey=True)
cost_order = list(METHODS)
runtime = costs['seconds']['mean'].reindex(cost_order)/60
memory = costs['peak_gpu_gb']['mean'].reindex(cost_order)
axes[0].barh(cost_order,runtime); axes[0].set_xlabel('Mean training minutes per final run')
axes[1].barh(cost_order,memory); axes[1].set_xlabel('Mean peak allocated GPU memory (GiB)')
from matplotlib.ticker import MaxNLocator
for ax in axes: ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
save_fig('11_training_cost')

def paired_cluster_bootstrap(predictions,test_df,repeats=2000):
    a,b = 'BanglaBERT_weighted_RDrop','BanglaBERT'
    seeds = sorted(set(predictions[predictions.method==a].seed)&set(predictions[predictions.method==b].seed))
    y = test_df.label.to_numpy()
    pa,pb = [],[]
    for seed in seeds:
        ra = predictions[(predictions.method==a)&(predictions.seed==seed)].set_index('id').loc[test_df.id]
        rb = predictions[(predictions.method==b)&(predictions.seed==seed)].set_index('id').loc[test_df.id]
        pa.append(ra.prediction.to_numpy()); pb.append(rb.prediction.to_numpy())
    groups = [np.array(v) for v in test_df.groupby('model_text',sort=False).indices.values()]
    rng = np.random.default_rng(2026); diffs=[]
    observed = np.mean([macro_f1(y,x)-macro_f1(y,z) for x,z in zip(pa,pb)])
    for _ in tqdm(range(repeats),desc='Paired cluster bootstrap'):
        idx = np.concatenate([groups[i] for i in rng.integers(0,len(groups),len(groups))])
        chosen_seeds = rng.integers(0,len(seeds),len(seeds))
        diffs.append(np.mean([macro_f1(y[idx],pa[s][idx])-macro_f1(y[idx],pb[s][idx]) for s in chosen_seeds]))
    lo,hi = np.quantile(diffs,[0.025,0.975])
    return dict(comparison=f'{a} minus {b}',observed_macro_f1_difference=float(observed),
                bootstrap_95ci=[float(lo),float(hi)],n_seeds=len(seeds),n_text_clusters=len(groups)),diffs

bootstrap_report,diffs = paired_cluster_bootstrap(predictions,test,BOOTSTRAP_REPEATS)
save_json(OUT/'tables/primary_bootstrap.json',bootstrap_report)
plt.figure(figsize=(7,4)); plt.hist(diffs,bins=40); plt.axvline(0,color='red',linestyle='--')
plt.xlabel('Macro-F1 difference: full method − BanglaBERT'); plt.ylabel('Bootstrap repetitions'); save_fig('10_paired_bootstrap')
print(bootstrap_report)


## 9. Literature context, error analysis, final report, reusable predictor
Published numbers below are source-reported reference values on different or
unverified protocols, not measured results from this notebook. SentNoB and
NC-SentNoB are not interchangeable. The 2024 score is micro-F1; the 2022 score
is macro-F1. No automatic claim of beating either paper is made.
Heuristic slices are not gold dialect/noise annotations. A fresh, independently
annotated test set and literature review remain necessary for broad novelty claims.


In [ ]:
literature = pd.DataFrame([
 {'paper':'BanglaBERT 2022','dataset':'SentNoB','model':'BanglaBERT','metric':'macro_f1','reported_score':0.7289,
  'url':'https://aclanthology.org/2022.findings-naacl.98/','directly_comparable':False},
 {'paper':'Elahi et al. 2024','dataset':'NC-SentNoB','model':'BanglaBERT original text','metric':'micro_f1','reported_score':0.75,
  'url':'https://aclanthology.org/2024.wnut-1.5/','directly_comparable':False},
 {'paper':'Elahi et al. 2024','dataset':'NC-SentNoB','model':'BanglaBERT back-translated','metric':'micro_f1','reported_score':0.72,
  'url':'https://aclanthology.org/2024.wnut-1.5/','directly_comparable':False}])
literature.to_csv(OUT/'tables/literature_context_NOT_head_to_head.csv',index=False)
primary = 'BanglaBERT_weighted_RDrop'
chosen_seed = int(chosen[chosen.method==primary].seed.iloc[0])
selected_run = next(r for r in final_runs if r['method']==primary and r['seed']==chosen_seed)
save_json(OUT/'selected_checkpoint.json',{'method':primary,'seed':chosen_seed,
    'path':str(Path(selected_run['path'])/'best_model'),'selection':'highest validation macro-F1 within predeclared method'})
rec = predictions[(predictions.method==primary)&(predictions.seed==chosen_seed)].copy()
rec['text'] = test.text.to_numpy()
rec['confidence'] = rec[['p0','p1','p2']].max(axis=1)
rec[rec.label!=rec.prediction].sort_values('confidence',ascending=False).to_csv(OUT/'tables/error_analysis.csv',index=False)
report_text = '\n'.join([
 '# Bangla sentiment experiment report',
 f'Execution mode: {MODE}. '+('SMOKE TEST ONLY; not research results.' if MODE=='smoke' else ''),
 f'Final split sizes: train={len(train)}, validation={len(val)}, test={len(test)}.',
 f'Exact-overlap cleaning enabled: {CLEAN_EXACT_OVERLAP}. Near-duplicate candidates: {len(near)}.',
 'Selection used validation macro-F1 only. Error bars are seed SD.',
 '## Measured results (0–1 scale)',summary.round(4).to_string(),
 '## Predeclared paired comparison',json.dumps(bootstrap_report,indent=2),
 '## Interpretation',
 'A positive estimate with a confidence interval above zero supports improvement over our plain BanglaBERT baseline on this evaluation.',
 'An interval overlapping zero is inconclusive. A negative estimate does not support the proposed improvement.',
 'Published scores are listed separately in literature_context_NOT_head_to_head.csv; split equivalence has not been established.',
 '## Limitations',
 'R-Drop is an existing technique. This notebook does not demonstrate methodological novelty.',
 'No gold noise annotations, human-checked spelling perturbations, or independent external test set were supplied.',
 'Contradictory training/dev groups were quarantined within each split; original test rows/labels, including contradictions, were retained.',
 'Near-duplicate candidates require review. Five seeds and bootstrap do not eliminate dataset bias.',
 'The model is not guaranteed to handle sarcasm, dialects, or unseen domains.',
 'All figures and tables are generated from saved predictions; no target score was inserted.',
 ])
(OUT/'REPORT.md').write_text(report_text,encoding='utf-8')

def predict_bangla(texts):
    checkpoint = Path(selected_run['path'])/'best_model'
    tok = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(DEVICE).eval()
    clean = [basic_normalize(normalize(t)) for t in texts]
    if any(not t for t in clean): raise ValueError('Input must contain nonempty text after normalization.')
    batch = tok(clean,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
    with torch.inference_mode(): probs = model(**batch).logits.softmax(-1).cpu().numpy()
    del model; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame({'text':texts,'prediction':[CLASS_NAMES[i] for i in probs.argmax(1)],'confidence':probs.max(1)})

print(predict_bangla(['সিনেমাটা ভালো লেগেছে।','সিনেমাটা ভালো লাগেনি।','আজ সোমবার।']).to_string(index=False))
# ZIP report/figures/predictions separately from large checkpoints to save disk.
import zipfile
archive = OUT.parent/'bangla_sentiment_report.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['figures','tables','audit']:
        for p in (OUT/folder).rglob('*'):
            if p.is_file(): z.write(p,p.relative_to(OUT))
    for name in ['REPORT.md','manifest.json','environment.txt','frozen_selection.json',
                 'resolved_model_revisions.json','selected_checkpoint.json']:
        z.write(OUT/name,name)
print('DONE. Report bundle:',archive)
print('Model checkpoints are separate under:',OUT/'runs')
print('Download outputs from Kaggle. Do not claim a perfect score or paper superiority without protocol verification.')
